# 任意本数ベルヌーイバンディッド実験  
## Random / Greedy / ε-Greedy の比較（平均だけでなくブレも可視化）

このノートブックでは、**腕の本数を任意に変更可能**なベルヌーイバンディッド実験を行う。  
さらに、**平均累積報酬・平均累積後悔**だけでなく、**runごとのブレ（ばらつき）**もグラフで可視化する。

## このノートブックで出す図
- 平均累積報酬（線のみ）
- 平均累積後悔（線のみ）
- **平均 ± 標準偏差** の帯つき累積報酬
- **平均 ± 標準偏差** の帯つき累積後悔
- **5–95 パーセンタイル帯** つき累積報酬
- **5–95 パーセンタイル帯** つき累積後悔

## 使い方
`TRUE_PROBS` を変えるだけで、何本腕でも実験できる。

例:
- 2本腕: `TRUE_PROBS = np.array([0.7, 0.5])`
- 3本腕: `TRUE_PROBS = np.array([0.7, 0.5, 0.4])`
- 5本腕: `TRUE_PROBS = np.array([0.7, 0.65, 0.55, 0.5, 0.3])`

## 注意
- **標準偏差帯**は「ブレの大きさ」をざっくり見るのに便利
- **パーセンタイル帯**は外れ値の影響を受けにくく、runの散らばりを見るのに自然

研究の文脈で「試行間のばらつき」を見せたいなら、**信頼区間よりも標準偏差帯や分位点帯の方が素直**なことが多い。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BASE_SEED = 42


## 実験設定

ここを変えれば、何本腕でも実験できる。


In [ ]:
# ====== ここを編集すれば腕の本数を変えられる ======
TRUE_PROBS = np.array([0.7, 0.5])   # 例: 2本腕
# TRUE_PROBS = np.array([0.7, 0.5, 0.4])   # 例: 3本腕
# TRUE_PROBS = np.array([0.7, 0.65, 0.55, 0.5, 0.3])   # 例: 5本腕
# ==========================================

N_ARMS = len(TRUE_PROBS)
N_STEPS = 1000
N_RUNS = 100
EPSILON = 0.1

if N_ARMS < 2:
    raise ValueError("腕は少なくとも2本必要です。")

if N_STEPS < N_ARMS:
    raise ValueError("N_STEPS は腕の本数以上にしてください。最初に全腕を1回ずつ引くためです。")

OPTIMAL_ARM = int(np.argmax(TRUE_PROBS))
OPTIMAL_VALUE = float(np.max(TRUE_PROBS))

print("TRUE_PROBS =", TRUE_PROBS)
print("N_ARMS =", N_ARMS)
print("N_STEPS =", N_STEPS)
print("N_RUNS =", N_RUNS)
print("EPSILON =", EPSILON)
print("OPTIMAL_ARM =", OPTIMAL_ARM)
print("OPTIMAL_VALUE =", OPTIMAL_VALUE)


## 補助関数

In [ ]:
def sample_reward(arm, rng, true_probs):
    """指定した腕のベルヌーイ報酬（0 or 1）を返す。"""
    return int(rng.random() < true_probs[arm])


def random_argmax(values, rng):
    """argmax で同値が複数ある場合にランダムで1つ選ぶ。"""
    max_value = np.max(values)
    candidates = np.flatnonzero(values == max_value)
    return rng.choice(candidates)


## 各方策の行動選択

In [ ]:
def select_action_random(q_values, rng, epsilon=None):
    return rng.integers(0, len(q_values))


def select_action_greedy(q_values, rng, epsilon=None):
    return random_argmax(q_values, rng)


def select_action_epsilon_greedy(q_values, rng, epsilon=0.1):
    if rng.random() < epsilon:
        return rng.integers(0, len(q_values))
    return random_argmax(q_values, rng)


## 1 run のシミュレーション

### 価値推定
各腕 \(a\) の推定価値 \(Q(a)\) はサンプル平均法で更新する。

\[
Q_{new}(a) = Q(a) + \frac{1}{N(a)}\left(r - Q(a)\right)
\]

### 初期条件
最初に**すべての腕を1回ずつ引く**。  
その後、残りの step で方策に従って行動選択する。


In [ ]:
POLICIES = {
    "Random": select_action_random,
    "Greedy": select_action_greedy,
    "Epsilon-Greedy": select_action_epsilon_greedy,
}


def run_single_bandit(policy_name, true_probs, n_steps=1000, epsilon=0.1, seed=None):
    rng = np.random.default_rng(seed)
    policy_fn = POLICIES[policy_name]

    n_arms = len(true_probs)
    optimal_value = float(np.max(true_probs))

    q_values = np.zeros(n_arms, dtype=float)
    counts = np.zeros(n_arms, dtype=int)
    reward_sums = np.zeros(n_arms, dtype=float)

    rewards = []
    regrets = []
    actions = []

    # ---- 初期化: すべての腕を1回ずつ引く ----
    for arm in range(n_arms):
        reward = sample_reward(arm, rng, true_probs)

        counts[arm] += 1
        reward_sums[arm] += reward
        q_values[arm] = reward_sums[arm] / counts[arm]

        rewards.append(reward)
        regrets.append(optimal_value - true_probs[arm])
        actions.append(arm)

    # ---- 残りの step を方策で進める ----
    for _ in range(n_arms, n_steps):
        if policy_name == "Epsilon-Greedy":
            action = policy_fn(q_values, rng, epsilon=epsilon)
        else:
            action = policy_fn(q_values, rng)

        reward = sample_reward(action, rng, true_probs)

        counts[action] += 1
        reward_sums[action] += reward
        q_values[action] = reward_sums[action] / counts[action]

        rewards.append(reward)
        regrets.append(optimal_value - true_probs[action])
        actions.append(action)

    rewards = np.array(rewards)
    regrets = np.array(regrets)
    actions = np.array(actions)

    cumulative_rewards = np.cumsum(rewards)
    cumulative_regrets = np.cumsum(regrets)

    return {
        "actions": actions,
        "rewards": rewards,
        "regrets": regrets,
        "cumulative_rewards": cumulative_rewards,
        "cumulative_regrets": cumulative_regrets,
        "final_q_values": q_values.copy(),
        "final_counts": counts.copy(),
    }


## 1 run だけ確認

In [ ]:
demo = run_single_bandit(
    policy_name="Greedy",
    true_probs=TRUE_PROBS,
    n_steps=N_STEPS,
    epsilon=EPSILON,
    seed=123
)

print("Final Q values:", demo["final_q_values"])
print("Final counts:", demo["final_counts"])
print("Final cumulative reward:", demo["cumulative_rewards"][-1])
print("Final cumulative regret:", demo["cumulative_regrets"][-1])


## 複数 run 実行して平均とブレを集計

In [ ]:
def run_multiple(policy_name, true_probs, n_runs=100, n_steps=1000, epsilon=0.1, base_seed=42):
    all_cum_rewards = []
    all_cum_regrets = []
    all_final_counts = []
    all_final_q_values = []

    for run_idx in range(n_runs):
        seed = base_seed + run_idx
        result = run_single_bandit(
            policy_name=policy_name,
            true_probs=true_probs,
            n_steps=n_steps,
            epsilon=epsilon,
            seed=seed
        )
        all_cum_rewards.append(result["cumulative_rewards"])
        all_cum_regrets.append(result["cumulative_regrets"])
        all_final_counts.append(result["final_counts"])
        all_final_q_values.append(result["final_q_values"])

    all_cum_rewards = np.array(all_cum_rewards)
    all_cum_regrets = np.array(all_cum_regrets)
    all_final_counts = np.array(all_final_counts)
    all_final_q_values = np.array(all_final_q_values)

    summary = {
        "all_cum_rewards": all_cum_rewards,
        "all_cum_regrets": all_cum_regrets,

        "mean_cum_rewards": all_cum_rewards.mean(axis=0),
        "std_cum_rewards": all_cum_rewards.std(axis=0),
        "p05_cum_rewards": np.percentile(all_cum_rewards, 5, axis=0),
        "p25_cum_rewards": np.percentile(all_cum_rewards, 25, axis=0),
        "p50_cum_rewards": np.percentile(all_cum_rewards, 50, axis=0),
        "p75_cum_rewards": np.percentile(all_cum_rewards, 75, axis=0),
        "p95_cum_rewards": np.percentile(all_cum_rewards, 95, axis=0),

        "mean_cum_regrets": all_cum_regrets.mean(axis=0),
        "std_cum_regrets": all_cum_regrets.std(axis=0),
        "p05_cum_regrets": np.percentile(all_cum_regrets, 5, axis=0),
        "p25_cum_regrets": np.percentile(all_cum_regrets, 25, axis=0),
        "p50_cum_regrets": np.percentile(all_cum_regrets, 50, axis=0),
        "p75_cum_regrets": np.percentile(all_cum_regrets, 75, axis=0),
        "p95_cum_regrets": np.percentile(all_cum_regrets, 95, axis=0),

        "mean_final_counts": all_final_counts.mean(axis=0),
        "mean_final_q_values": all_final_q_values.mean(axis=0),
        "final_reward_mean": all_cum_rewards[:, -1].mean(),
        "final_reward_std": all_cum_rewards[:, -1].std(),
        "final_regret_mean": all_cum_regrets[:, -1].mean(),
        "final_regret_std": all_cum_regrets[:, -1].std(),
    }
    return summary


In [ ]:
results = {}

for policy_name in POLICIES.keys():
    results[policy_name] = run_multiple(
        policy_name=policy_name,
        true_probs=TRUE_PROBS,
        n_runs=N_RUNS,
        n_steps=N_STEPS,
        epsilon=EPSILON,
        base_seed=BASE_SEED * 1000
    )

print("実行完了")
list(results.keys())


## 最終結果の数値確認

In [ ]:
for policy_name, summary in results.items():
    print("=" * 70)
    print(policy_name)
    print(f"Final cumulative reward mean ± std: {summary['final_reward_mean']:.3f} ± {summary['final_reward_std']:.3f}")
    print(f"Final cumulative regret mean ± std: {summary['final_regret_mean']:.3f} ± {summary['final_regret_std']:.3f}")
    print(f"Mean final action counts: {summary['mean_final_counts']}")
    print(f"Mean final Q values: {summary['mean_final_q_values']}")


## 図1: 平均累積報酬（線のみ）

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(1, N_STEPS + 1)

for policy_name, summary in results.items():
    plt.plot(x, summary["mean_cum_rewards"], label=policy_name)

plt.xlabel("Step")
plt.ylabel("Average Cumulative Reward")
plt.title("Average Cumulative Reward over Steps")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 図2: 平均累積後悔（線のみ）

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(1, N_STEPS + 1)

for policy_name, summary in results.items():
    plt.plot(x, summary["mean_cum_regrets"], label=policy_name)

plt.xlabel("Step")
plt.ylabel("Average Cumulative Regret")
plt.title("Average Cumulative Regret over Steps")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 図3: 平均 ± 標準偏差の帯つき累積報酬

これは「平均的な傾向」に加えて、**run間のブレの大きさ**を見る図。


In [ ]:
def plot_mean_std_band(results, metric_mean_key, metric_std_key, ylabel, title):
    plt.figure(figsize=(10, 6))
    x = np.arange(1, N_STEPS + 1)

    for policy_name, summary in results.items():
        mean = summary[metric_mean_key]
        std = summary[metric_std_key]

        plt.plot(x, mean, label=policy_name)
        plt.fill_between(x, mean - std, mean + std, alpha=0.2)

    plt.xlabel("Step")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_mean_std_band(
    results,
    metric_mean_key="mean_cum_rewards",
    metric_std_key="std_cum_rewards",
    ylabel="Cumulative Reward",
    title="Cumulative Reward with Mean ± Std"
)


## 図4: 平均 ± 標準偏差の帯つき累積後悔

In [ ]:
plot_mean_std_band(
    results,
    metric_mean_key="mean_cum_regrets",
    metric_std_key="std_cum_regrets",
    ylabel="Cumulative Regret",
    title="Cumulative Regret with Mean ± Std"
)


## 図5: 5–95 パーセンタイル帯つき累積報酬

こちらは、**runの分布の広がり**を見るのに向いている。  
外れ値の影響を受けにくいので、こちらの方が好みの人も多い。


In [ ]:
def plot_percentile_band(results, mean_key, p_low_key, p_high_key, ylabel, title):
    plt.figure(figsize=(10, 6))
    x = np.arange(1, N_STEPS + 1)

    for policy_name, summary in results.items():
        mean = summary[mean_key]
        p_low = summary[p_low_key]
        p_high = summary[p_high_key]

        plt.plot(x, mean, label=policy_name)
        plt.fill_between(x, p_low, p_high, alpha=0.2)

    plt.xlabel("Step")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_percentile_band(
    results,
    mean_key="mean_cum_rewards",
    p_low_key="p05_cum_rewards",
    p_high_key="p95_cum_rewards",
    ylabel="Cumulative Reward",
    title="Cumulative Reward with 5–95 Percentile Band"
)


## 図6: 5–95 パーセンタイル帯つき累積後悔

In [ ]:
plot_percentile_band(
    results,
    mean_key="mean_cum_regrets",
    p_low_key="p05_cum_regrets",
    p_high_key="p95_cum_regrets",
    ylabel="Cumulative Regret",
    title="Cumulative Regret with 5–95 Percentile Band"
)


## さらに細かく見たい場合

25–75 パーセンタイル帯（四分位範囲）を出すと、中央50%の散らばりだけを見られる。  
図がややすっきりするので、本文ではこちらの方が使いやすいこともある。


In [ ]:
plot_percentile_band(
    results,
    mean_key="mean_cum_rewards",
    p_low_key="p25_cum_rewards",
    p_high_key="p75_cum_rewards",
    ylabel="Cumulative Reward",
    title="Cumulative Reward with 25–75 Percentile Band"
)


## 例: 3本腕に変えるには

```python
TRUE_PROBS = np.array([0.7, 0.5, 0.4])
```

## 例: 10本腕に変えるには

```python
TRUE_PROBS = np.array([0.9, 0.82, 0.8, 0.75, 0.71, 0.6, 0.55, 0.4, 0.35, 0.2])
```


## 解釈メモ

- **平均だけ**を見ると、Greedy が意外と悪く見えないことがある
- しかし**帯つきグラフ**を見ると、Greedy は run によって当たり外れが大きいことが分かる
- つまり、「平均性能」と「安定性」は別物である

このへんが見えると、結果節が少し論文っぽくなる。数字だけでなく、**分布の顔つき**が見えるからだ。
